<a href="https://colab.research.google.com/github/rburchf1/AI102Challenges/blob/main/assignment1_sentiment_ryan_burchfield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 — Sentiment Analysis: Classical ML vs. DL vs. LLM

**Ryan Burchfield**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Overview & dataset selection

Twitter US Airline Sentiment

I selected the Twitter dataset because the sentiment analysis contains 3 options: positive, negative, and neutral. Determining neutrality seems more challenging, in terms of semantic analysis, than a binary setup. In addition, the tertiary framework is more comparable to the type of analysis I might conduct at work.

## 2. Load & inspect data
Use Hugging Face `datasets` or local CSV to load your dataset. Show class distribution and a few samples.

In [4]:
# TODO: Install/Import basics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# from utils.data_utils import basic_clean # Commented out, as 'utils' module is not found
print('OK: libraries imported')

OK: libraries imported


In [6]:
# Load Twitter Dataset
file_path = '/content/drive/MyDrive/Tweets.csv'
try:
  ds = pd.read_csv(file_path)
  print('File loaded successfully.')
except FileNotFoundError:
  print(f'File not found at path: {file_path}. Please check the file path.')
except Exception as e:
  print(f'An error occurred while loading the file: {e}')

#Class distribution and a few examples
print("DataSet Info:")
ds.info()
print("\nDataSet Description:")
display(ds.describe(include='all'))
print("\nFirst 5 rows of the DataSet:")
display(ds.head())

File loaded successfully.
DataSet Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created       

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
count,1.464000e+04,14640,14640.000000,9178,10522.000000,14640,40,14640,32,14640.000000,14640,1019,14640,9907,9820
unique,NaN,3,NaN,10,NaN,6,3,7701,13,NaN,14427,832,14247,3081,85
top,NaN,negative,NaN,Customer Service Issue,NaN,United,negative,JetBlueNews,Customer Service Issue,NaN,@united thanks,"[0.0, 0.0]",2015-02-24 09:54:34 -0800,"Boston, MA",Eastern Time (US & Canada)
freq,NaN,9178,NaN,2910,NaN,3822,32,63,12,NaN,6,164,5,157,3744
mean,5.692184e+17,NaN,0.900169,NaN,0.638298,NaN,NaN,NaN,NaN,0.082650,NaN,NaN,NaN,NaN,NaN
std,7.791112e+14,NaN,0.162830,NaN,0.330440,NaN,NaN,NaN,NaN,0.745778,NaN,NaN,NaN,NaN,NaN
min,5.675883e+17,NaN,0.335000,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
25%,5.685592e+17,NaN,0.692300,NaN,0.360600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
50%,5.694779e+17,NaN,1.000000,NaN,0.670600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
75%,5.698905e+17,NaN,1.000000,NaN,1.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN



First 5 rows of the DataSet:


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [7]:
# Evaluate the dataset for balance

x = ds['text']
y = ds['airline_sentiment']

print('Sentiment class distribution:')
display(y.value_counts())

Sentiment class distribution:


,count
airline_sentiment,
negative,9178
neutral,3099
positive,2363


COMMENTS ON DATASET BALANCE

As one might expect, the dataset is imbalanced towared the negative. It seems that customers are most likely to spend effort to tweet an experience when it is negative and accept a neutral or good experience as a fair trade since the paid for a service.

## 3. Preprocessing
Start simple: lowercase, remove URLs/usernames (if any), strip spaces, and tokenize if needed. Consider whether to keep emojis.

In [10]:
#EMOJIS
#Detect if there are emojies in the dataset

import re

# Regex to detect most common emojis
# This regex covers a broad range of Unicode emoji blocks and sequences.
# It might not catch all edge cases or very new emojis, but it's a good starting point.
emoji_pattern = re.compile(
    "["  # Start character set
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Miscellaneous Symbols and Pictographs
    "\U0001F680-\U0001F6FF"  # Transport and Map Symbols
    "\U0001F1E0-\U0001F1FF"  # Regional indicator symbols
    "\U00002600-\U000026FF"  # Miscellaneous Symbols
    "\U00002700-\U000027BF"  # Dingbats
    "]+"
)

def contains_emoji(text):
    return bool(emoji_pattern.search(str(text)))

# Sample some texts to check for emojis
print("Checking for emojis in sample texts:")
found_emoji = False
for i, text in enumerate(x.sample(n=10, random_state=42)):
    if contains_emoji(text):
        print(f"  Text {i+1} (contains emoji): {text}")
        found_emoji = True
    else:
        print(f"  Text {i+1} (no emoji): {text}")

if not found_emoji:
    print("  No emojis found in the selected sample of texts.")
else:
    print("  Emojis were found in the sample texts.")

print("\nNow checking the full dataset (this might take a moment if the dataset is large):")
# Check the entire 'text' column for emojis
num_texts_with_emojis = x.apply(contains_emoji).sum()

if num_texts_with_emojis > 0:
    print(f"The 'text' column contains emojis. Found {num_texts_with_emojis} texts with emojis.")
    print("It would be beneficial to decide whether to remove or preserve them during preprocessing based on the task.")
else:
    print("The 'text' column does not appear to contain emojis.")

Checking for emojis in sample texts:
  Text 1 (no emoji): @SouthwestAir you're my early frontrunner for best airline! #oscars2016
  Text 2 (no emoji): @USAirways how is it that my flt to EWR was Cancelled Flightled yet flts to NYC from USAirways are still flying?
  Text 3 (no emoji): @JetBlue what is going on with your BDL to DCA flights yesterday and today?! Why is every single one getting delayed?
  Text 4 (no emoji): @JetBlue do they have to depart from Washington, D.C.??
  Text 5 (no emoji): @JetBlue I can probably find some of them. Are the ticket #s on there?
  Text 6 (no emoji): @united still waiting to hear back. My wallet was stolen from one of your planes so would appreciate a resolution here
  Text 7 (no emoji): @united Yes my flight was rebooked. I'm just losing trust in you if I want to get anywhere on time.
  Text 8 (no emoji): @JetBlue Thank you ! What about Paris ? Could we arrange something from there ?
  Text 9 (no emoji): @united not 100% sure, however my ticket incl

EMOJIS ANALYSIS AND DECISION ON WHETHER TO KEEP OR REMOVE EMOJIS WHEN CLEANING DATA

The dataset contains 487 entries with emojis, which represent only ~3% of the tweets. Therefore, emojis will be removed.

In [8]:
# Split data for training and validation and testing
from sklearn.model_selection import train_test_split

#FIRST SPLIT
#80% training + validation, 20% for test

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#SECOND SPLIT
#Take 12.5% of the 80% for validation
#12.5% of 80% = 10% of the original dataset

x_train, x_val, y_train, y_val = train_test_split(
    x, y,
    test_size=0.125,
    random_state=42,
    stratify=y
)

In [11]:
#Define and apply a clean function

def basic_clean(text):
    text = str(text).lower()  # Convert to string and lowercase

    # Remove URLs (http/https links)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove mentions (@usernames)
    text = re.sub(r'@\w+', '', text)

    # Remove emojis using the previously defined pattern
    text = emoji_pattern.sub(r'', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning function
clean_x_train = x_train.apply(basic_clean)
clean_x_val = x_val.apply(basic_clean)
clean_x_test = x_test.apply(basic_clean)

print("Original text sample:", x_train.iloc[0])
print("Cleaned text sample:", clean_x_train.iloc[0])

print("\nFirst 5 cleaned training texts:")
display(clean_x_train.head())

Original text sample: @VirginAmerica Can't bring up my reservation online using Flight Booking Problems code
Cleaned text sample: can't bring up my reservation online using flight booking problems code

First 5 cleaned training texts:


,text
86,can't bring up my reservation online using fli...
14047,educate bohol is a 501(c)(3) w/all volunteer s...
3642,i mean is there a real live person somewhere i...
2356,how about plowing the snow at a gate before th...
5455,i met my twitter friend waiting outside the tr...


In [12]:
#Additional preprocessing: toeknization, remove stopwords, remove puncutation, stemming, and lemmatization

import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
import string

nltk.download('stopwords')
nltk.download('wordnet')

text = "Natural Language Processing enables computers to understand human language."

ext = text.translate(str.maketrans('', '', string.punctuation))

# Tokenize (split) the text into separate words.
# text.split() breaks the sentence into a list of words using spaces.
tokens = text.split()

# Get the set of English stop words.
# A set is used because it allows fast checking of "is this word in the list?"
stop_words = set(stopwords.words('english'))

# Remove stop words from our list of tokens.
# This list comprehension means:
# "for each word in tokens, keep it only if it is NOT in stop_words".
filtered_tokens = [word for word in tokens if word not in stop_words]

# Create (initialize) the stemmer and lemmatizer objects that we will use.
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Apply stemming and then lemmatization to each filtered word.
# For each word in filtered_tokens:
#   1. stemmer.stem(word) gets the stem (basic form)
#   2. lemmatizer.lemmatize(...) changes that stem to a dictionary form if possible
# The final results are stored in a new list called processed_tokens.
processed_tokens = [lemmatizer.lemmatize(stemmer.stem(word)) for word in filtered_tokens]

# Print the cleaned version of the text (lowercased and without punctuation)
print("Original Text:")
print(text)

# Print a blank line and then the tokens after removing stop words
print("\nFiltered Tokens (Stop Words Removed):")
print(filtered_tokens)

# Print another blank line and then the tokens after stemming and lemmatization
print("\nProcessed Tokens (Stemmed and Lemmatized):")
print(processed_tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Original Text:
Natural Language Processing enables computers to understand human language.

Filtered Tokens (Stop Words Removed):
['Natural', 'Language', 'Processing', 'enables', 'computers', 'understand', 'human', 'language.']

Processed Tokens (Stemmed and Lemmatized):
['natur', 'languag', 'process', 'enabl', 'comput', 'understand', 'human', 'language.']


## 4. Model A: Classical ML (TF–IDF + Logistic Regression or SVM)

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2))
xtr = tfidf.fit_transform(x_train)
xva = tfidf.transform(x_val)
clf = LogisticRegression(max_iter=200)
clf.fit(xtr, y_train)
preds = clf.predict(xva)

print(classification_report(y_val, preds, digits=4))


              precision    recall  f1-score   support

    negative     0.8166    0.9590    0.8821      1147
     neutral     0.7182    0.5387    0.6156       388
    positive     0.8333    0.5424    0.6571       295

    accuracy                         0.8027      1830
   macro avg     0.7894    0.6800    0.7183      1830
weighted avg     0.7985    0.8027    0.7893      1830



## 5. Model B: Deep Learning (LSTM)

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import numpy as np

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Label Encoding
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# 2. Text Tokenization and Padding
max_words = 10000  # Vocabulary size
max_len = 100    # Max sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(clean_x_train)

x_train_seq = tokenizer.texts_to_sequences(clean_x_train)
x_val_seq = tokenizer.texts_to_sequences(clean_x_val)
x_test_seq = tokenizer.texts_to_sequences(clean_x_test)

x_train_padded = pad_sequences(x_train_seq, maxlen=max_len)
x_val_padded = pad_sequences(x_val_seq, maxlen=max_len)
x_test_padded = pad_sequences(x_test_seq, maxlen=max_len)

# 3. PyTorch Dataset and DataLoader
BATCH_SIZE = 64

train_data = TensorDataset(torch.LongTensor(x_train_padded), torch.LongTensor(y_train_encoded))
val_data = TensorDataset(torch.LongTensor(x_val_padded), torch.LongTensor(y_val_encoded))
test_data = TensorDataset(torch.LongTensor(x_test_padded), torch.LongTensor(y_test_encoded))

train_loader = DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_data, shuffle=False, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, shuffle=False, batch_size=BATCH_SIZE)

# 4. LSTM Model Definition
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers, bidirectional, dropout_rate):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout_rate if num_layers > 1 else 0, # Dropout only if num_layers > 1
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), output_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, text):
        # text = [batch size, seq len]
        embedded = self.dropout(self.embedding(text))
        # embedded = [batch size, seq len, embedding dim]

        output, (hidden, cell) = self.lstm(embedded)
        # output = [batch size, seq len, hidden dim * num directions]
        # hidden = [num layers * num directions, batch size, hidden dim]

        # Use the hidden state from the last layer (and combine directions if bidirectional)
        if self.lstm.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
        # hidden = [batch size, hidden dim * num directions]

        return self.fc(hidden)

# Model parameters
VOCAB_SIZE = max_words # From tokenizer
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
OUTPUT_DIM = len(label_encoder.classes_) # Number of unique sentiment classes
NUM_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT_RATE = 0.5

model = SentimentLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, NUM_LAYERS, BIDIRECTIONAL, DROPOUT_RATE).to(device)

# 5. Training Loop
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters())

def train(model, loader, optimizer, criterion):
    model.train()
    epoch_loss = 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            predictions = model(texts)
            loss = criterion(predictions, labels)
            epoch_loss += loss.item()

            all_preds.extend(predictions.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return epoch_loss / len(loader), all_preds, all_labels

N_EPOCHS = 3 # Train for 3 epochs

print("\nStarting LSTM Training...")
for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss, _, _ = evaluate(model, val_loader, criterion)
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}')

# 6. Evaluation on Test Set
print("\nEvaluating on Test Set...")
test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion)

print(f'Test Loss: {test_loss:.3f}')

# Convert numerical predictions and labels back to original sentiment for reporting
original_test_labels = label_encoder.inverse_transform(test_labels)
original_test_preds = label_encoder.inverse_transform(test_preds)

print("\nClassification Report for LSTM Model:")
print(classification_report(original_test_labels, original_test_preds, digits=4))

accuracy = accuracy_score(original_test_labels, original_test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(original_test_labels, original_test_preds, average='weighted', zero_division=0)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Precision (weighted): {precision:.4f}")
print(f"Recall (weighted): {recall:.4f}")
print(f"F1-Score (weighted): {f1:.4f}")


Using device: cpu

Starting LSTM Training...
Epoch: 01 | Train Loss: 0.796 | Val Loss: 0.701
Epoch: 02 | Train Loss: 0.677 | Val Loss: 0.601
Epoch: 03 | Train Loss: 0.616 | Val Loss: 0.594

Evaluating on Test Set...
Test Loss: 0.569

Classification Report for LSTM Model:
              precision    recall  f1-score   support

    negative     0.8568    0.8316    0.8440      1835
     neutral     0.5425    0.7000    0.6113       620
    positive     0.7695    0.5645    0.6512       473

    accuracy                         0.7606      2928
   macro avg     0.7229    0.6987    0.7022      2928
weighted avg     0.7762    0.7606    0.7636      2928


Accuracy: 0.7606
Precision (weighted): 0.7762
Recall (weighted): 0.7606
F1-Score (weighted): 0.7636


Starting LSTM Training...
Epoch: 01 | Train Loss: 0.796 | Val Loss: 0.701
Epoch: 02 | Train Loss: 0.677 | Val Loss: 0.601
Epoch: 03 | Train Loss: 0.616 | Val Loss: 0.594

Evaluating on Test Set...
Test Loss: 0.569

Classification Report for LSTM Model:
              precision    recall  f1-score   support

    negative     0.8568    0.8316    0.8440      1835
     neutral     0.5425    0.7000    0.6113       620
    positive     0.7695    0.5645    0.6512       473

    accuracy                         0.7606      2928
   macro avg     0.7229    0.6987    0.7022      2928
weighted avg     0.7762    0.7606    0.7636      2928


Accuracy: 0.7606
Precision (weighted): 0.7762
Recall (weighted): 0.7606
F1-Score (weighted): 0.7636

In [15]:
!pip install keras

## 6. Model C: LLM (BERT/DistilBERT fine-tuning)

In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
import pandas as pd # Import pandas

# Re-encode labels for Hugging Face Trainer compatibility, ensuring it matches the model output_dim
# The previous label_encoder is for PyTorch, so we need to ensure consistency or re-do if necessary
# Assuming y_train, y_val, y_test are still Series with original labels.

# Ensure labels are integers starting from 0 for the Hugging Face model
# First, combine all labels to fit the encoder, then transform subsets
all_labels = pd.concat([y_train, y_val, y_test])
label_encoder_hf = LabelEncoder()
label_encoder_hf.fit(all_labels)

train_labels_encoded = label_encoder_hf.transform(y_train)
val_labels_encoded = label_encoder_hf.transform(y_val)
test_labels_encoded = label_encoder_hf.transform(y_test)

num_classes = len(label_encoder_hf.classes_)
print(f"Number of classes: {num_classes}")

tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# Correcting variable names and using encoded labels
train_ds = Dataset.from_dict({'text': clean_x_train.tolist(), 'label': train_labels_encoded.tolist()})
val_ds  = Dataset.from_dict({'text': clean_x_val.tolist(), 'label': val_labels_encoded.tolist()})
test_ds  = Dataset.from_dict({'text': clean_x_test.tolist(),  'label': test_labels_encoded.tolist()})

def tokenize(b):
  # Removed padding=True here, let the Trainer's DataCollatorWithPadding handle it dynamically
  return tok(b['text'], truncation=True, max_length=256)

ds_tok = DatasetDict({
    'train': train_ds.map(tokenize, batched=True),
    'validation': val_ds.map(tokenize, batched=True),
    'test': test_ds.map(tokenize, batched=True)
})

# Correcting num_labels to use the dynamically determined number of classes
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_classes)

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)
  p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
  acc = accuracy_score(labels, preds)
  return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}

args = TrainingArguments(output_dir='./out', num_train_epochs=2, per_device_train_batch_size=16, per_device_eval_batch_size=32, eval_strategy='epoch', learning_rate=2e-5, weight_decay=0.01)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['validation'], # Use validation set for evaluation during training
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tok) # Explicitly provide data collator
)

print("\nStarting LLM Fine-tuning...")
trainer.train()

print("\nEvaluating LLM on Test Set...")
llm_eval_results = trainer.evaluate(eval_dataset=ds_tok['test'])
print(llm_eval_results)

# Optionally, extract and print individual metrics for comparison
llm_accuracy = llm_eval_results.get('eval_accuracy')
llm_precision = llm_eval_results.get('eval_precision')
llm_recall = llm_eval_results.get('eval_recall')
llm_f1 = llm_eval_results.get('eval_f1')

print(f"\nLLM Test Accuracy: {llm_accuracy:.4f}")
print(f"LLM Test Precision (weighted): {llm_precision:.4f}")
print(f"LLM Test Recall (weighted): {llm_recall:.4f}")
print(f"LLM Test F1-Score (weighted): {llm_f1:.4f}")

Number of classes: 3


Map:   0%|          | 0/12810 [00:00<?, ? examples/s]

Map:   0%|          | 0/1830 [00:00<?, ? examples/s]

Map:   0%|          | 0/2928 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Evaluating LLM on Test Set...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.327063,0.370994,2,0.865779,0.864933,0.865779,0.865296


{'eval_loss': 0.3709944486618042, 'eval_accuracy': 0.8657786885245902, 'eval_precision': 0.864933206610755, 'eval_recall': 0.8657786885245902, 'eval_f1': 0.8652961764894624}

LLM Test Accuracy: 0.8658
LLM Test Precision (weighted): 0.8649
LLM Test Recall (weighted): 0.8658
LLM Test F1-Score (weighted): 0.8653


Epoch	Training Loss	Validation Loss	Accuracy	Precision	Recall	F1
1	0.541662	0.454034	0.821311	0.827214	0.821311	0.823739
2	0.327063	0.451944	0.832240	0.831132	0.832240	0.831586

In [20]:
!pip install transformers accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.9 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


## 7. Results comparison
Create a table of metrics for all models (Accuracy/Precision/Recall/F1).

In [20]:
import pandas as pd

# --- TF-IDF + Logistic Regression Metrics (from cell xACEQdZsETry output) ---
# From the output:
#              precision    recall  f1-score   support
#    negative     0.8166    0.9590    0.8821      1147
#     neutral     0.7182    0.5387    0.6156       388
#    positive     0.8333    0.5424    0.6571       295
#    accuracy                         0.8027      1830
#   macro avg     0.7894    0.6800    0.7183      1830
# weighted avg     0.7985    0.8027    0.7893      1830

tfidf_lr_accuracy = 0.8027
tfidf_lr_precision = 0.7985 # weighted avg
tfidf_lr_recall = 0.8027   # weighted avg (same as accuracy in this report)
tfidf_lr_f1 = 0.7893     # weighted avg

# --- LSTM Model Metrics (from cell tUJoWXkdKs6y output) ---
# From the output:
# Accuracy: 0.7606
# Precision (weighted): 0.7762
# Recall (weighted): 0.7606
# F1-Score (weighted): 0.7636

lstm_accuracy = 0.7606
lstm_precision = 0.7762
lstm_recall = 0.7606
lstm_f1 = 0.7636

# --- DistilBERT Model Metrics (from llm_eval_results in kernel state) ---
# The variables `llm_accuracy`, `llm_precision`, `llm_recall`, `llm_f1` are available in the kernel state
distilbert_accuracy = llm_accuracy
distilbert_precision = llm_precision
distilbert_recall = llm_recall
distilbert_f1 = llm_f1

# Create a DataFrame for comparison
results = pd.DataFrame({
    'Model': ['TF-IDF + Logistic Regression', 'LSTM', 'DistilBERT'],
    'Accuracy': [tfidf_lr_accuracy, lstm_accuracy, distilbert_accuracy],
    'Precision (weighted)': [tfidf_lr_precision, lstm_precision, distilbert_precision],
    'Recall (weighted)': [tfidf_lr_recall, lstm_recall, distilbert_recall],
    'F1-Score (weighted)': [tfidf_lr_f1, lstm_f1, distilbert_f1]
})

print("\n--- Model Comparison ---")
display(results.set_index('Model').round(4))


--- Model Comparison ---


,Accuracy,Precision (weighted),Recall (weighted),F1-Score (weighted)
Model,,,,
TF-IDF + Logistic Regression,0.8027,0.7985,0.8027,0.7893
LSTM,0.7606,0.7762,0.7606,0.7636
DistilBERT,0.8658,0.8649,0.8658,0.8653


Model
TF-IDF + Logistic Regression 0.8027 0.7893
LSTM 0.7606 0.7636
DistilBERT 0.8658 0.8653
From these results, it's clear that the DistilBERT model performed the best across all metrics (Accuracy, Precision, Recall, and F1-Score). It achieved an accuracy of approximately 86.58%.

The TF-IDF + Logistic Regression model came in second, with an accuracy of around 80.27%.

The LSTM model had the lowest performance among the three, with an accuracy of approximately 76.06%.

This trend is generally expected, as pre-trained large language models like DistilBERT often capture more nuanced semantic information, leading to better performance in text classification tasks compared to traditional machine learning (TF-IDF + Logistic Regression) or simpler deep learning models (LSTM) when fine-tuned on specific datasets.


## 8. Error analysis & reflection
Show misclassified examples and discuss patterns. Connect to challenges (context, emojis, domain shift, explainability).

## 9. Reproducibility notes (how to run)
List environment, hardware (CPU/GPU), and exact commands used.